# Statistical Analysis


In [ ]:
%pip install pandas numpy matplotlib seaborn scipy statsmodels gdown

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')


## 1. Load Data


In [ ]:
import gdown
df = pd.read_csv(gdown.download(id='1a-MNVGzc5DV-MTDmkWOCP5qhVTaaZMnu', quiet=False))
df.head()


## 2. Hypothesis Testing: T-Test

**Hypothesis:** Does offering a discount significantly affect the average profit?
*   **Null Hypothesis ($H_0$):** There is no significant difference in average profit between discounted and non-discounted orders.
*   **Alternative Hypothesis ($H_a$):** There is a significant difference in average profit.


In [ ]:
# --- 1. INDEPENDENT TWO-SAMPLE T-TEST ---
# Objective: Determine if applying a discount significantly impacts average profit.
# Step A: Split the data into two groups: Orders with discounts, and orders without.
discounted_profit = df[df['discount'] > 0]['profit']
non_discounted_profit = df[df['discount'] == 0]['profit']

# Step B: Perform Welch's t-test (equal_var=False assumes the two groups may have different variances)
t_stat, p_value = stats.ttest_ind(discounted_profit, non_discounted_profit, equal_var=False)

print(f'T-statistic: {t_stat:.4f}')
print(f'P-value: {p_value:.4e}')

# Step C: Interpret the results against our significance level (Alpha = 0.05)
# If p-value < 0.05, it means the difference is statistically significant and not due to random chance.
alpha = 0.05
if p_value < alpha:
    print('Reject the null hypothesis: There is a significant difference in average profit.')
else:
    print('Fail to reject the null hypothesis: There is no significant difference in average profit.')


## 3. Hypothesis Testing: ANOVA

**Hypothesis:** Is there a significant difference in average revenue across different product categories?
*   **Null Hypothesis ($H_0$):** The average revenue is the same across all product categories.
*   **Alternative Hypothesis ($H_a$):** At least one product category has a significantly different average revenue.


In [ ]:
# --- 2. ANOVA (Analysis of Variance) ---
# Objective: Determine if the average revenue differs significantly between multiple product categories.
# Step A: Fit an Ordinary Least Squares (OLS) regression model treating 'category' as a categorical independent variable.
model = ols('revenue ~ C(category)', data=df).fit()
# Step B: Run a Type-2 ANOVA on the fitted model to generate the F-statistic and P-value
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

# Step C: Evaluate the P-value
p_val_anova = anova_table['PR(>F)'].iloc[0]
if p_val_anova < 0.05:
    print('\nReject the null hypothesis: Significant difference in revenue across categories.')
else:
    print('\nFail to reject the null hypothesis.')


## 4. Hypothesis Testing: Chi-Square Test

**Hypothesis:** Is there an association between a customer's gender and the store type they visit?
*   **Null Hypothesis ($H_0$):** Gender and Store Type are independent.
*   **Alternative Hypothesis ($H_a$):** Gender and Store Type are associated.


In [ ]:
# --- 3. CHI-SQUARE TEST OF INDEPENDENCE ---
# Objective: Determine if there is a relationship between Customer Gender and the Store Type they visit.
# Step A: Create a contingency table counting frequencies of each gender/store_type combination.
contingency_table = pd.crosstab(df['gender'], df['store_type'])
print('Contingency Table:')
print(contingency_table)

# Step B: Compute the Chi-Square statistic to compare expected frequencies vs observed frequencies.
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

print(f'\nChi-Square Statistic: {chi2:.4f}')
print(f'P-value: {p:.4e}')

# Step C: Evaluate the result
if p < 0.05:
    print('Reject the null hypothesis: There is an association between Gender and Store Type.')
else:
    print('Fail to reject the null hypothesis: Gender and Store Type are independent.')


## 5. Pearson Correlation

Let's check the correlation specifically between 'discount' and 'profit' to quantify their relationship.


In [ ]:
# --- 4. PEARSON CORRELATION TEST ---
# Objective: Determine if there is a statistically significant linear correlation between Discount and Profit.
# Note: A correlation of 0 means no linear relationship, 1 means perfect positive, -1 means perfect negative.
corr_coef, p_val_corr = stats.pearsonr(df['discount'], df['profit'])

print(f'Pearson Correlation Coefficient: {corr_coef:.4f}')
print(f'P-value: {p_val_corr:.4e}')

# Step B: Check if the observed correlation is statistically significant (p < 0.05)
if p_val_corr < 0.05:
    print('There is a statistically significant correlation between discount and profit.')
else:
    print('The correlation is not statistically significant.')
